# Beslisbomen
In de vorige lessen hebben we al veel bouwstenen van het beslsiboom algoritme gemaakt. Je ziet de bouwstenen hieronder. 
- Voer onderstaande cellen uit.

In [2]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
%matplotlib inline
from sklearn import tree as tree_plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

cancer = load_breast_cancer()
df = pd.DataFrame(cancer.data, columns = cancer.feature_names)
df['IsBenign'] = cancer.target

print( {n: v for n, v in zip(['kwaadaardig', 'goedaardig'], np.bincount(cancer.target))})
print('1 in kolom IsBenign betekent dat de tumor goedaardig is')

X = df.drop('IsBenign', axis=1)
y = df['IsBenign']

X_train, X_test, y_train, y_test = train_test_split(X,y, random_state = 42)

{'kwaadaardig': np.int64(212), 'goedaardig': np.int64(357)}
1 in kolom IsBenign betekent dat de tumor goedaardig is


In [3]:
import pandas as pd
import numpy as np

# Functie om aantallen van labels te tellen
def count_values(y):
    counts = {}
    for value in y:
        if value in counts:
            counts[value] += 1
        else:
            counts[value] = 1
    return counts

# Geeft de meest voorkomende klasse terug
def max_value(y):
    counts = count_values(y)
    max_value = 0
    max_key = None
    for key, value in counts.items():
        if value > max_value:
            max_value = value
            max_key = key
    return max_key

# Berekenen van de Gini-index
def gini_index(y):
    counts = count_values(y)
    total = sum(counts.values())
    gini_index = 1 - sum((count / total) ** 2 for count in counts.values())
    return gini_index, total

# Berekenen van de entropie
def entropy(y):
    counts = count_values(y)
    total = sum(counts.values())
    entropy = - sum((count / total) * np.log2(count / total) for count in counts.values())
    return entropy, total

# Gewogen impurity score berekenen
def weighted(left_node, right_node, metric):
    if metric == 'gini':
        left_metric, left_total = gini_index(left_node)
        right_metric, right_total = gini_index(right_node)
    elif metric == 'entropy':
        left_metric, left_total = entropy(left_node)
        right_metric, right_total = entropy(right_node)

    total = left_total + right_total
    weighted_metric = (left_total / total) * left_metric + (right_total / total) * right_metric
    return weighted_metric


class BinaryTree:
    def __init__(self, value):
        self.value = value
        self.left = None
        self.right = None

    def create_left_node(self, value):
        self.left = BinaryTree(value)              
        
    def create_right_node(self, value):
        self.right = BinaryTree(value)        

    def get_right(self):        
        return self.right       

    def get_left(self):       
        return self.left
    
class DecisionTree(BinaryTree):
    def __init__(self, X, y, depth=0, value=None):

        # Hier zien we iets nieuws:
        # DecisionTree erft van BinaryTree.
        # Door super().__init__(value) aan te roepen,
        # krijgt DecisionTree automatisch de eigenschappen
        # van BinaryTree mee (left, right, value).
        # In eerdere lessen maakten we meestal losse classes
        # zonder inheritance/overerving.

        super().__init__(value)

        self.depth = depth 
        self.X = X
        self.y = y

        # Deze variabelen zijn voorbereid voor het echte
        # decision tree algoritme.
        self.feature = None
        self.threshold = None
        self.samples = None        
        self.value = None
        self.metric = None
        self.score = None
        self.prediction = None        
        self.left = None
        self.right = None         

        # Wat hebben we al af?
        # - Functies voor impurity berekeningen (gini, entropy)
        # - Functie voor gewogen score
        # - Structuur van een binaire boom
        # - Predict-functie om door de boom te lopen
        # - Opslag van eigenschappen van een node

        # Wat moeten we nog maken?
        # - Het trainen/fitten van de boom
        # - Het zoeken naar de beste split
        # - Het bepalen van feature en threshold
        # - Het recursief maken van nieuwe nodes
        # - Stopcriteria (max depth, minimum samples, pure node)
        # - Leaf nodes een prediction geven


def dt_predict_row(tree, row):

    # Als een node geen kinderen heeft,
    # dan is het een leaf node.
    if tree.left is None and tree.right is None:
        return tree.prediction

    X_value = row[tree.feature]

    if X_value < tree.threshold:

        # Ga recursief naar links
        return dt_predict_row(tree.get_left(), row)

    else:

        # Ga recursief naar rechts
        return dt_predict_row(tree.get_right(), row)
    
def predict(tree, X):

    # Voorspel voor elke rij in X
    return [dt_predict_row(tree, row) for _, row in X.iterrows()]

### 1 Beslisboom algoritme
- Bestudeer bovenstaande cel en vergelijk dit met het decision tree algoritme dat we in eerdere lessen hebben uitgeschreven. Welke onderdelen van het algoritme hebben we nu af en welke moeten we nog maken? Gebruik comments in bovenstaande cel om antwoord te geven.
- Wat valt je op aan de wijze waarop de class DecisionTree is aangemaakt? In welk opzicht is dit anders dan in eerdere lessen? Gebruik comments in bovenstaande cel om antwoord te geven.

### 2 Bepalen van de beste splitsing.

Je hebt eerder een functie weighted(left_y, right_y, metric) geïmplementeerd die de gewogen score (bijvoorbeeld Gini of entropie) berekent voor een splitsing van de doelvariabele y.

In deze opdracht ga je een functie schrijven die bepaalt welke feature en welke splitwaarde de dataset het best opdeelt, op basis van die gewogen metric.

- Schrijf een functie best_split(X, y, metric) die de beste feature en bijbehorende splitwaarde selecteert door alle mogelijke splitsingen te evalueren.

- Parameters:
    - X: een pandas DataFrame met de features / onafhankelijke variabelen.
    - y: een pandas Series of DataFrame met de doelvariabele / afhankelijke variabele.
    - metric: een string, 'gini' of 'entropy'.

- Output: De functie retourneert een tuple met (best_feature (string), best_value (float), best_left_X (df), best_left_y (df), best_right_X (df), best_right_y (df)). 

**Stappen**:
1) Loop over alle kolommen (features) in X.
2) Loop over alle unieke waarden in die kolom (mogelijke splitpunten). Binnen de loop:
    - Splits X en y in twee delen: links (≤ waarde) en rechts (> waarde).
    - Bereken de gewogen score van de splitsing met je eerder gemaakte weighted()-functie.
    - Bewaar de gegevens van beste splitsing (laagste score):
        - best_feature: de feature waarmee de beste splitsing wordt bereikt.
        - best_value: waarde uit de kolom waarmee de beste splitsing wordt bereikt.
        - best_left_X: het dataframe met features dat in aan het linker kind van de beslisboom wordt toegevoegd in de volgende functie die je gaat definieren.
        - best_left_y: het dataframe met de target dat in aan het linker kind van de beslisboom wordt toegevoegd in de volgende functie die je gaat definieren.
        - best_right_X: het dataframe met features dat in aan het rechter kind van de beslisboom wordt toegevoegd in de volgende functie die je gaat definieren.
        - best_right_y: het dataframe met de target dat in aan het rechter kind van de beslisboom wordt toegevoegd in de volgende functie die je gaat definieren.
Na de loop:
6) Return de gegevens van de beste splitsing.

**Tips**: 
- Gebruik np.unique() om alle mogelijke splitwaarden voor een kolom te vinden. Dit zorgt ervoor dat alle unieke waardes in een kolom worden gesorteerd van laag naar hoog.
- Gebruik masking zoals X[col] <= val om subsets te maken.

In [ ]:
def best_split(X, y, metric):
    best_feature = None
    best_score = np.inf



    return best_feature, best_value, best_left_X, best_left_y, best_right_X, best_right_y


best_split(X_train, y_train, 'gini')

In [4]:
def best_split(X, y, metric):

    best_feature = None
    best_value = None
    best_score = np.inf

    best_left_X = None
    best_left_y = None
    best_right_X = None
    best_right_y = None

    # Loop over alle features
    for col in X.columns:

        # Zoek alle unieke mogelijke splitwaarden
        unique_values = np.unique(X[col])

        # Loop over alle mogelijke splitwaarden
        for val in unique_values:

            # Maak masks
            left_mask = X[col] <= val
            right_mask = X[col] > val

            # Split X
            left_X = X[left_mask]
            right_X = X[right_mask]

            # Split y
            left_y = y[left_mask]
            right_y = y[right_mask]

            # Sla lege splitsingen over
            if len(left_y) == 0 or len(right_y) == 0:
                continue

            # Bereken gewogen score
            score = weighted(left_y, right_y, metric)

            # Kijk of dit de beste split tot nu toe is
            if score < best_score:

                best_score = score
                best_feature = col
                best_value = val

                best_left_X = left_X
                best_left_y = left_y

                best_right_X = right_X
                best_right_y = right_y

    return (
        best_feature,
        best_value,
        best_left_X,
        best_left_y,
        best_right_X,
        best_right_y
    )


# Test
best_split(X_train, y_train, 'gini')

('mean concave points',
 np.float64(0.05074),
      mean radius  mean texture  mean perimeter  mean area  mean smoothness  \
 287       12.890         13.12           81.89      515.9          0.06955   
 402       12.960         18.29           84.18      525.2          0.07351   
 184       15.280         22.41           98.92      710.6          0.09057   
 442       13.780         15.79           88.37      585.9          0.08817   
 54        15.100         22.02           97.26      712.8          0.09056   
 ..           ...           ...             ...        ...              ...   
 20        13.080         15.71           85.63      520.0          0.10750   
 71         8.888         14.64           58.79      244.0          0.09783   
 106       11.640         18.33           75.17      412.5          0.11420   
 270       14.290         16.82           90.30      632.6          0.06429   
 102       12.180         20.52           77.22      458.7          0.08013   
 
    

### 3 Decision Tree bouwen

Schrijf een functie `build_tree(X, y, metric, depth=0, max_depth=10)` die een decision tree opbouwt op basis van de trainingsdata.


**Parameters**:
- `X`: een pandas DataFrame met de invoervariabelen.  
- `y`: een pandas Series met de doelvariabele.  
- `metric`: een string `'gini'` of `'entropy'` voor de impurity-metric.  
- `depth`: het huidige niveau van de node (standaard = 0).  
- `max_depth`: de maximale diepte van de boom.  

**Output**:   
- Een `DecisionTree` object (de root van de boom), dat eventueel subtrees bevat in `.left` en `.right`.

**Stappen:**

1. Maak een nieuwe node aan met de DecisionTree class:
2. Sla informatie op in de attributen van de node:
    - samples: aantal rijen in y
    - value: aantal maal dat iedere klasse voorkomt in y (gebruik count_values())
    - prediction: meest voorkomende klasse (gebruik max_value())
    - score: impurity score op basis van de gekozen metric (gebruik gini_index() of entropy())
3. Controleer stopcriteria (=base case):
    - Is de diepte ≥ max_depth?
    - Is de impurity score gelijk aan 0?
    - Zijn er minder dan 2 samples?
    - In deze gevallen return je de node.
4. Bepaal de beste splitsing (gebruik best_split()) en sla best_feature en best_value op in de attributen feature en threshold van de node.
5. Bouw recursief de linker en rechter subboom.
6. Return de node.

In [ ]:
def build_tree(X, y, metric, depth=0, max_depth=10):

    return node

# Voorbeeld van het aanroepen van de functie
tree = build_tree(X_train, y_train, 'gini')




In [ ]:
def build_tree(X, y, metric, depth=0, max_depth=10):

    # Maak een nieuwe node
    node = DecisionTree(X, y, depth=depth)

    # Sla informatie op in de node
    node.samples = len(y)
    node.value = count_values(y)
    node.prediction = max_value(y)
    node.metric = metric

    # Bereken impurity score
    if metric == 'gini':
        node.score = gini_index(y)[0]

    elif metric == 'entropy':
        node.score = entropy(y)[0]

    # Maximale diepte bereikt
    if depth >= max_depth:
        return node

    # Pure node
    if node.score == 0:
        return node

    # Te weinig samples
    if len(y) < 2:
        return node

    (
        best_feature,
        best_value,
        best_left_X,
        best_left_y,
        best_right_X,
        best_right_y
    ) = best_split(X, y, metric)

    # Als geen geldige split gevonden wordt
    if best_feature is None:
        return node

    # Sla splitinformatie op
    node.feature = best_feature
    node.threshold = best_value

    node.left = build_tree(
        best_left_X,
        best_left_y,
        metric,
        depth=depth + 1,
        max_depth=max_depth
    )

    node.right = build_tree(
        best_right_X,
        best_right_y,
        metric,
        depth=depth + 1,
        max_depth=max_depth
    )

    return node

# Voorbeeld van het aanroepen van de functie
tree = build_tree(X_train, y_train, 'gini')

### 4 Print de boom
- Gebruik de printfunctie van vorige les om de tree te visualiseren.

In [7]:
def print_tree(tree, depth=0):

    if tree is None:
        return

    indent = "--->" * depth
    print(f"{indent}{tree.value}")

    print_tree(tree.get_left(), depth + 1)
    print_tree(tree.get_right(), depth + 1)

print(print_tree(tree))

{1: 268, 0: 158}
--->{1: 251, 0: 16}
--->--->{1: 244, 0: 5}
--->--->--->{1: 243, 0: 3}
--->--->--->--->{1: 243, 0: 2}
--->--->--->--->--->{1: 5, 0: 1}
--->--->--->--->--->--->{1: 5}
--->--->--->--->--->--->{0: 1}
--->--->--->--->--->{1: 238, 0: 1}
--->--->--->--->--->--->{1: 225}
--->--->--->--->--->--->{1: 13, 0: 1}
--->--->--->--->--->--->--->{0: 1}
--->--->--->--->--->--->--->{1: 13}
--->--->--->--->{0: 1}
--->--->--->{0: 2, 1: 1}
--->--->--->--->{1: 1}
--->--->--->--->{0: 2}
--->--->{0: 11, 1: 7}
--->--->--->{1: 5}
--->--->--->{0: 11, 1: 2}
--->--->--->--->{0: 11}
--->--->--->--->{1: 2}
--->{0: 142, 1: 17}
--->--->{1: 10, 0: 4}
--->--->--->{1: 10}
--->--->--->{0: 4}
--->--->{0: 138, 1: 7}
--->--->--->{0: 6, 1: 7}
--->--->--->--->{1: 7}
--->--->--->--->{0: 6}
--->--->--->{0: 132}
None


### 5 Vergelijk de beslisboom met die van SciKit learn.
Hieronder wordt de beslisboom van sklearn getoond die is getraind (.fit) op dezelfde data als onze boom. 
- Wat zijn de verschillen en overeenkomsten?
- Wat zijn mogelijke verklaringen voor de verschillen?

In [10]:
sk_tree = DecisionTreeClassifier(random_state = 42, max_depth=10)
sk_tree.fit(X_train, y_train)

fig = plt.figure(figsize=(140,60))
_ = tree_plt.plot_tree(sk_tree, 
                   feature_names=list(cancer.feature_names),   
                   class_names=list(cancer.target_names),
                   filled=True, impurity = True, fontsize = 60)

plt.show()

### Voorspellingen vergelijken
- Vergelijk de voorspellingen met de eigen beslisboom met die van Sklearn.
